# 1회차 스터디 - LangGraph ReAct 에이전트

## 실행 순서
1. Anthropic API 키 발급 : https://console.anthropic.com
2. 셀 순서대로 실행
3. 마지막 셀에서 명령어 바꿔가며 테스트

In [ ]:
# 0. 패키지 설치
!pip install langgraph langchain-anthropic langchain-core -q

In [ ]:
# 1. API 키 설정
import os
from google.colab import userdata

# Colab 왼쪽 사이드바 자물쇠 아이콘 -> Secrets -> ANTHROPIC_API_KEY 추가
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
# 2. VideoContext 스키마 정의
from typing import Annotated, TypedDict, Optional

class Scene(TypedDict):
    start: float
    end: float
    description: str

class Transcript(TypedDict):
    start: float
    end: float
    text: str

class VideoContext(TypedDict):
    file_path: str
    duration: float
    scenes: list[Scene]
    transcript: list[Transcript]

print("VideoContext 스키마 정의 완료")

In [ ]:
# 3. State 정의
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    video_context: Optional[VideoContext]
    edit_history: list[str]

print("AgentState 정의 완료")

In [ ]:
# 4. Tool 정의
import json
from langchain_core.tools import tool

@tool
def search_scene(query: str) -> str:
    """장면 검색 Tool - 나중에 실제 VideoContext 검색으로 교체"""
    return json.dumps({
        "query": query,
        "result": [
            {"start": 10.0, "end": 25.0, "description": f"'{query}' 관련 장면 더미"}
        ]
    }, ensure_ascii=False)


@tool
def calculate_timestamp(expression: str) -> str:
    """타임스탬프 계산 Tool - 나중에 실제 영상 시간 계산으로 교체"""
    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": result})
    except Exception as e:
        raise ValueError(f"계산 실패 : {str(e)}")


@tool
def get_video_info(file_path: str) -> str:
    """영상 정보 조회 Tool - 나중에 FFmpeg 메타데이터 추출로 교체"""
    return json.dumps({
        "file_path": file_path,
        "duration": 120.0,
        "resolution": "1920x1080",
        "fps": 30
    })


tools = [search_scene, calculate_timestamp, get_video_info]
tool_map = {t.name: t for t in tools}

print(f"Tool 등록 완료 : {list(tool_map.keys())}")

In [ ]:
# 5. LLM + Node + Graph 정의
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-3-5-sonnet-20241022")
llm_with_tools = llm.bind_tools(tools)


def agent_node(state: AgentState) -> AgentState:
    messages = state["messages"]

    if state.get("video_context"):
        context_str = json.dumps(state["video_context"], ensure_ascii=False)
        system_prompt = f"현재 편집 중인 영상 정보:\n{context_str}"
        messages = [HumanMessage(content=system_prompt)] + messages

    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def tool_node(state: AgentState) -> AgentState:
    last_message = state["messages"][-1]
    tool_results = []
    edit_history = state.get("edit_history", [])

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        try:
            result = tool_map[tool_name].invoke(tool_args)
            edit_history.append(f"{tool_name} : 성공")
            tool_results.append(
                ToolMessage(content=str(result), tool_call_id=tool_call["id"])
            )
        except Exception as e:
            error_msg = f"{tool_name} 실행 실패 : {str(e)}"
            edit_history.append(f"{tool_name} : 실패")
            tool_results.append(
                ToolMessage(content=error_msg, tool_call_id=tool_call["id"])
            )

    return {"messages": tool_results, "edit_history": edit_history}


def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool"
    return END


def build_graph():
    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tool", tool_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue, {"tool": "tool", END: END})
    graph.add_edge("tool", "agent")
    return graph.compile()


print("Graph 빌드 완료")

In [ ]:
# 6. 실행 함수
def run_agent_stream(user_input: str, video_context=None):
    graph = build_graph()

    initial_state = {
        "messages": [HumanMessage(content=user_input)],
        "video_context": video_context,
        "edit_history": []
    }

    print(f"\n[입력] {user_input}")
    print("-" * 60)

    for step in graph.stream(initial_state):
        for node_name, state in step.items():
            print(f"\n>> {node_name}")

            for message in state["messages"]:
                if isinstance(message, AIMessage):
                    if message.content:
                        print(f"   AI : {message.content}")
                    if hasattr(message, "tool_calls") and message.tool_calls:
                        for tc in message.tool_calls:
                            print(f"   Tool 선택 : {tc['name']}")
                            print(f"   Args      : {tc['args']}")
                elif isinstance(message, ToolMessage):
                    print(f"   Tool 결과 : {message.content}")

            if state.get("edit_history"):
                print(f"   편집 기록 : {state['edit_history']}")

    print("\n" + "-" * 60)

print("실행 함수 정의 완료")

In [ ]:
# 7. 테스트 실행
# 여기 user_input 바꿔가면서 테스트해봐

run_agent_stream("sample.mp4 영상 정보 조회하고 10 + 25 계산해줘")

In [ ]:
# 8. 숙제 - 여기에 Tool 하나 추가해봐

@tool
def my_tool(input: str) -> str:
    """내가 만든 Tool - 기능은 자유"""
    # 여기 구현
    return f"my_tool 실행 결과 : {input}"


# tools 리스트에 추가
tools.append(my_tool)
tool_map[my_tool.name] = my_tool

# 추가된 Tool로 다시 실행
llm_with_tools = llm.bind_tools(tools)

print(f"Tool 목록 : {list(tool_map.keys())}")